In [1]:
# Cell 1 - Loading the knowledge base
import pandas as pd
import json

print("Loading knowledge base...")

# Load the full knowledge base you already generated
kb_df = pd.DataFrame()

try:
    with open('knowledge_base_full.json', 'r') as f:
        kb_data = json.load(f)
    kb_df = pd.DataFrame(kb_data)
    print(f"✅ Loaded knowledge base with {len(kb_df)} nouns")
    print(f"✅ Properties per noun: {len(kb_df.columns) - 1}")  # -1 for 'noun' column
except FileNotFoundError:
    print("❌ knowledge_base_full.json not found!")
    print("Please make sure you've generated the knowledge base first (Cell 12)")

# Display summary
print(f"\nKnowledge Base Overview:")
print(f"Total nouns: {len(kb_df)}")
print(f"\nFirst 5 nouns:")
print(kb_df['noun'].head())

print(f"\nSample data for '{kb_df['noun'].iloc[0]}':")
print(kb_df.iloc[0])

Loading knowledge base...
✅ Loaded knowledge base with 1852 nouns
✅ Properties per noun: 25

Knowledge Base Overview:
Total nouns: 1852

First 5 nouns:
0         aardvark
1             acai
2    action figure
3            actor
4            adobe
Name: noun, dtype: object

Sample data for 'aardvark':
noun                       aardvark
is_living                      True
is_animal                      True
is_plant                      False
is_object                     False
is_abstract                   False
is_natural                     True
is_artificial                 False
size_scale_1_10                 3.0
weight_scale_1_10               3.0
speed_scale_1_10                NaN
danger_to_humans_1_10           1.0
fragility_scale_1_10            NaN
is_edible                     False
rarity_1_10                     7.0
typical_indoor_use_1_10         1.0
portability_1_10                1.0
tech_complexity_1_10            1.0
can_fly                       False
lives_in_water

In [2]:
# Cell 15 - Question Selection Algorithm

import numpy as np

class TwentyQuestionsGame:
    """
    20 Questions game using property-based filtering
    """

    def __init__(self, knowledge_base_df):
        """
        Initialize with the knowledge base
        """
        self.kb = knowledge_base_df.copy()
        self.all_nouns = set(self.kb['noun'].values)
        self.properties = [col for col in self.kb.columns if col != 'noun']

        # Separate binary and scale properties
        self.binary_props = [p for p in self.properties
                            if self.kb[p].dtype == bool or
                            set(self.kb[p].dropna().unique()).issubset({True, False, 1, 0})]
        self.scale_props = [p for p in self.properties if p not in self.binary_props]

        print(f"Game initialized with {len(self.all_nouns)} nouns")
        print(f"Binary properties: {len(self.binary_props)}")
        print(f"Scale properties: {len(self.scale_props)}")

    def calculate_information_gain(self, remaining_nouns, property_name, threshold=None):
        """
        Calculate how well a question splits the remaining nouns
        Higher score = better question
        """
        if len(remaining_nouns) <= 1:
            return 0

        subset = self.kb[self.kb['noun'].isin(remaining_nouns)]

        if property_name in self.binary_props:
            # Binary question: how evenly does it split?
            true_count = subset[property_name].sum()
            false_count = len(subset) - true_count

            # Best split is 50/50
            smaller = min(true_count, false_count)
            larger = max(true_count, false_count)

            if larger == 0:
                return 0

            # Return ratio - closer to 1.0 is better (more even split)
            return smaller / larger if larger > 0 else 0

        elif property_name in self.scale_props and threshold is not None:
            # Scale question with threshold
            above = (subset[property_name] > threshold).sum()
            below = len(subset) - above

            smaller = min(above, below)
            larger = max(above, below)

            return smaller / larger if larger > 0 else 0

        return 0

    def find_best_question(self, remaining_nouns):
        """
        Find the question that best splits the remaining nouns
        """
        if len(remaining_nouns) <= 1:
            return None, None, None

        best_score = 0
        best_property = None
        best_threshold = None
        best_question_text = None

        # Try binary properties
        for prop in self.binary_props:
            score = self.calculate_information_gain(remaining_nouns, prop)

            if score > best_score:
                best_score = score
                best_property = prop
                best_threshold = None
                best_question_text = self._format_binary_question(prop)

        # Try scale properties with different thresholds
        for prop in self.scale_props:
            subset = self.kb[self.kb['noun'].isin(remaining_nouns)]
            values = subset[prop].dropna()

            if len(values) > 0:
                # Try median as threshold
                median = values.median()
                score = self.calculate_information_gain(remaining_nouns, prop, median)

                if score > best_score:
                    best_score = score
                    best_property = prop
                    best_threshold = median
                    best_question_text = self._format_scale_question(prop, median)

        return best_property, best_threshold, best_question_text

    def _format_binary_question(self, property_name):
        """Format a binary property as a yes/no question"""
        questions = {
            'is_living': "Is it a living thing?",
            'is_animal': "Is it an animal?",
            'is_plant': "Is it a plant?",
            'is_object': "Is it a physical object?",
            'is_abstract': "Is it an abstract concept?",
            'is_natural': "Is it found in nature?",
            'is_artificial': "Is it man-made?",
            'can_fly': "Can it fly?",
            'lives_in_water': "Does it live in water?",
            'is_predator': "Is it a predator?",
            'requires_electricity': "Does it require electricity?",
            'is_food': "Is it food?",
            'is_vehicle': "Is it a vehicle?",
            'is_tool': "Is it a tool?",
            'is_furniture': "Is it furniture?",
            'is_edible': "Is it edible?",
        }
        return questions.get(property_name, f"Is it {property_name}?")

    def _format_scale_question(self, property_name, threshold):
        """Format a scale property as a comparison question"""
        questions = {
            'size_scale_1_10': f"Is it larger than {threshold:.1f}/10 in size?",
            'weight_scale_1_10': f"Is it heavier than {threshold:.1f}/10 in weight?",
            'speed_scale_1_10': f"Is it faster than {threshold:.1f}/10 in speed?",
            'danger_to_humans_1_10': f"Is it more dangerous than {threshold:.1f}/10?",
            'fragility_scale_1_10': f"Is it more fragile than {threshold:.1f}/10?",
            'rarity_1_10': f"Is it rarer than {threshold:.1f}/10?",
            'typical_indoor_use_1_10': f"Is it more commonly found indoors than {threshold:.1f}/10?",
            'portability_1_10': f"Is it more portable than {threshold:.1f}/10?",
            'tech_complexity_1_10': f"Is it more technologically complex than {threshold:.1f}/10?",
            'edibleness': f"Is it more edible than {threshold:.1f}/10?",
        }
        return questions.get(property_name, f"Is {property_name} greater than {threshold:.1f}?")

    def filter_nouns(self, remaining_nouns, property_name, threshold, answer):
        """
        Filter nouns based on the answer to a question
        """
        subset = self.kb[self.kb['noun'].isin(remaining_nouns)]

        if threshold is None:  # Binary property
            if answer:
                filtered = subset[subset[property_name] == True]
            else:
                filtered = subset[subset[property_name] == False]
        else:  # Scale property
            if answer:
                filtered = subset[subset[property_name] > threshold]
            else:
                filtered = subset[subset[property_name] <= threshold]

        return set(filtered['noun'].values)

print("✅ TwentyQuestionsGame class defined")

✅ TwentyQuestionsGame class defined


In [3]:
# Cell 16 - PLAY GAME FUNCTION
import random

def play_game_interactive(game, target_noun, max_questions=20, verbose=True):
    """
    Play a game of 20 questions interactively

    Args:
        game: TwentyQuestionsGame instance
        target_noun: The noun to guess
        max_questions: Maximum questions allowed
        verbose: Print detailed output

    Returns:
        dict with game results
    """
    if target_noun not in game.all_nouns:
        return {
            'success': False,
            'error': f"'{target_noun}' not in knowledge base"
        }

    remaining_nouns = game.all_nouns.copy()
    questions_asked = []
    answers_given = []

    if verbose:
        print("="*70)
        print(f"🎮 PLAYING 20 QUESTIONS")
        print(f"Target noun: {target_noun}")
        print("="*70)

    for question_num in range(1, max_questions + 1):
        if verbose:
            print(f"\n--- Question {question_num} ---")
            print(f"Remaining possibilities: {len(remaining_nouns)}")

        # Check if we're down to one noun
        if len(remaining_nouns) == 1:
            final_guess = list(remaining_nouns)[0]
            questions_asked.append(f"Are you thinking of {final_guess}?")

            if verbose:
                print(f"\nQ{question_num}: Are you thinking of {final_guess}?")
                print(f"A: {'YES! ✅' if final_guess == target_noun else 'NO ❌'}")

            return {
                'success': final_guess == target_noun,
                'final_guess': final_guess,
                'target_noun': target_noun,
                'questions_asked': question_num,
                'questions': questions_asked,
                'answers': answers_given + [final_guess == target_noun],
                'remaining_nouns': 1
            }

        # Last question must be a guess
        if question_num == max_questions:
            # Make best guess from remaining nouns
            final_guess = list(remaining_nouns)[0]  # Could be smarter here
            questions_asked.append(f"Are you thinking of {final_guess}?")

            if verbose:
                print(f"\nQ{question_num}: Are you thinking of {final_guess}? (Final guess)")
                print(f"A: {'YES! ✅' if final_guess == target_noun else 'NO ❌'}")

            return {
                'success': final_guess == target_noun,
                'final_guess': final_guess,
                'target_noun': target_noun,
                'questions_asked': question_num,
                'questions': questions_asked,
                'answers': answers_given + [final_guess == target_noun],
                'remaining_nouns': len(remaining_nouns)
            }

        # Find best question
        prop, threshold, question_text = game.find_best_question(remaining_nouns)

        if prop is None:
            # No good question found, make a guess
            final_guess = list(remaining_nouns)[0]
            questions_asked.append(f"Are you thinking of {final_guess}?")

            if verbose:
                print(f"\nQ{question_num}: Are you thinking of {final_guess}? (Forced guess)")
                print(f"A: {'YES! ✅' if final_guess == target_noun else 'NO ❌'}")

            return {
                'success': final_guess == target_noun,
                'final_guess': final_guess,
                'target_noun': target_noun,
                'questions_asked': question_num,
                'questions': questions_asked,
                'answers': answers_given + [final_guess == target_noun],
                'remaining_nouns': len(remaining_nouns)
            }

        # Get the correct answer from knowledge base
        target_row = game.kb[game.kb['noun'] == target_noun].iloc[0]

        if threshold is None:  # Binary
            correct_answer = bool(target_row[prop])
        else:  # Scale
            correct_answer = float(target_row[prop]) > threshold

        questions_asked.append(question_text)
        answers_given.append(correct_answer)

        if verbose:
            print(f"\nQ{question_num}: {question_text}")
            print(f"A: {'YES' if correct_answer else 'NO'}")

        # Filter remaining nouns
        remaining_nouns = game.filter_nouns(remaining_nouns, prop, threshold, correct_answer)

        if len(remaining_nouns) == 0:
            if verbose:
                print("\n❌ ERROR: No remaining nouns! Knowledge base inconsistency.")
            return {
                'success': False,
                'error': 'No remaining nouns',
                'questions_asked': question_num,
                'questions': questions_asked,
                'answers': answers_given
            }

    # Should not reach here
    return {
        'success': False,
        'error': 'Max questions reached',
        'questions_asked': max_questions,
        'remaining_nouns': len(remaining_nouns)
    }

# Initialize game
game = TwentyQuestionsGame(kb_df)

# Test with one noun
test_noun = random.choice(list(game.all_nouns))
print(f"\n\n🧪 Testing with: {test_noun}\n")
result = play_game_interactive(game, test_noun, verbose=True)

print(f"\n\n{'='*70}")
print("GAME RESULT:")
print(f"Success: {result['success']}")
print(f"Questions used: {result.get('questions_asked', 'N/A')}")
print("="*70)

Game initialized with 1852 nouns
Binary properties: 16
Scale properties: 9


🧪 Testing with: band press

🎮 PLAYING 20 QUESTIONS
Target noun: band press

--- Question 1 ---
Remaining possibilities: 1852

Q1: Is it more dangerous than 1.0/10?
A: YES

--- Question 2 ---
Remaining possibilities: 816

Q2: Is it heavier than 3.0/10 in weight?
A: YES

--- Question 3 ---
Remaining possibilities: 360

Q3: Is it more commonly found indoors than 2.0/10?
A: YES

--- Question 4 ---
Remaining possibilities: 167

Q4: Is it faster than 5.0/10 in speed?
A: YES

--- Question 5 ---
Remaining possibilities: 66

Q5: Is it more portable than 4.0/10?
A: YES

--- Question 6 ---
Remaining possibilities: 32

Q6: Is it more portable than 8.0/10?
A: NO

--- Question 7 ---
Remaining possibilities: 17

Q7: Is it a physical object?
A: YES

--- Question 8 ---
Remaining possibilities: 9

Q8: Is it man-made?
A: NO

--- Question 9 ---
Remaining possibilities: 3

Q9: Is it a living thing?
A: NO

--- Question 10 ---
Remai

In [4]:
print("="*70)
print("RUNNING 5 TEST GAMES")
print("="*70)

# Manually enter the 5 nouns that you want to play
instructor_nouns = [
    "amazon",
    "bicycle",
    "computer",
    "apple",
    "ocean"
]

print(f"\nTest nouns: {instructor_nouns}\n")

# Verify all nouns are in knowledge base
missing_nouns = [n for n in instructor_nouns if n not in game.all_nouns]
if missing_nouns:
    print(f"⚠️  WARNING: These nouns are not in the knowledge base: {missing_nouns}")
    print("The game will fail for these nouns.\n")

# Run the games
results = []

for i, noun in enumerate(instructor_nouns, 1):
    if noun not in game.all_nouns:
        print(f"\n{'#'*70}")
        print(f"GAME {i}/5: {noun}")
        print(f"{'#'*70}")
        print(f"❌ ERROR: '{noun}' not found in knowledge base")
        results.append({
            'success': False,
            'target_noun': noun,
            'error': 'Not in knowledge base'
        })
        continue

    print(f"\n\n{'#'*70}")
    print(f"GAME {i}/5: {noun}")
    print(f"{'#'*70}")

    result = play_game_interactive(game, noun, verbose=True)
    results.append(result)

    print(f"\n✅ Game {i} complete")

print(f"\n\n{'='*70}")
print("ALL 5 GAMES COMPLETE")
print("="*70)

RUNNING 5 TEST GAMES

Test nouns: ['amazon', 'bicycle', 'computer', 'apple', 'ocean']



######################################################################
GAME 1/5: amazon
######################################################################
🎮 PLAYING 20 QUESTIONS
Target noun: amazon

--- Question 1 ---
Remaining possibilities: 1852

Q1: Is it more dangerous than 1.0/10?
A: NO

--- Question 2 ---
Remaining possibilities: 864

Q2: Is it more portable than 7.0/10?
A: NO

--- Question 3 ---
Remaining possibilities: 419

Q3: Is it more portable than 2.0/10?
A: YES

--- Question 4 ---
Remaining possibilities: 194

Q4: Is it more portable than 5.0/10?
A: YES

--- Question 5 ---
Remaining possibilities: 91

Q5: Is it found in nature?
A: NO

--- Question 6 ---
Remaining possibilities: 55

Q6: Is it more commonly found indoors than 8.0/10?
A: NO

--- Question 7 ---
Remaining possibilities: 31

Q7: Is it more technologically complex than 2.0/10?
A: NO

--- Question 8 ---
Remaining possibil